<a href="https://colab.research.google.com/github/Vaishnavi-Bhujbal/Data-Replication-Simulator/blob/main/Distributed_Database_Sharding_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pandas plotly streamlit

In [2]:
import pandas as pd

# Create sample customer data
data = []

for i in range(1001, 1301):
    data.append({
        "customer_id": i,
        "name": f"Customer_{i}",
        "city": ["Pune", "Mumbai", "Nashik", "Nagpur"][i % 4],
        "email": f"customer{i}@gmail.com",
        "balance": (i * 10) % 50000
    })

# Convert into DataFrame
df = pd.DataFrame(data)

# Save dataset
df.to_csv("customers_sample_300.csv", index=False)

# Display dataset
print("Dataset created successfully!")
print("Total rows and columns:", df.shape)

df.head(10)

Dataset created successfully!
Total rows and columns: (300, 5)


,customer_id,name,city,email,balance
0,1001,Customer_1001,Mumbai,customer1001@gmail.com,10010
1,1002,Customer_1002,Nashik,customer1002@gmail.com,10020
2,1003,Customer_1003,Nagpur,customer1003@gmail.com,10030
3,1004,Customer_1004,Pune,customer1004@gmail.com,10040
4,1005,Customer_1005,Mumbai,customer1005@gmail.com,10050
5,1006,Customer_1006,Nashik,customer1006@gmail.com,10060
6,1007,Customer_1007,Nagpur,customer1007@gmail.com,10070
7,1008,Customer_1008,Pune,customer1008@gmail.com,10080
8,1009,Customer_1009,Mumbai,customer1009@gmail.com,10090
9,1010,Customer_1010,Nashik,customer1010@gmail.com,10100


In [3]:
print("Dataset shape:", df.shape)
display(df.head(10))
print("Total customers:", len(df))

Dataset shape: (300, 5)


,customer_id,name,city,email,balance
0,1001,Customer_1001,Mumbai,customer1001@gmail.com,10010
1,1002,Customer_1002,Nashik,customer1002@gmail.com,10020
2,1003,Customer_1003,Nagpur,customer1003@gmail.com,10030
3,1004,Customer_1004,Pune,customer1004@gmail.com,10040
4,1005,Customer_1005,Mumbai,customer1005@gmail.com,10050
5,1006,Customer_1006,Nashik,customer1006@gmail.com,10060
6,1007,Customer_1007,Nagpur,customer1007@gmail.com,10070
7,1008,Customer_1008,Pune,customer1008@gmail.com,10080
8,1009,Customer_1009,Mumbai,customer1009@gmail.com,10090
9,1010,Customer_1010,Nashik,customer1010@gmail.com,10100


Total customers: 300


In [4]:
import sqlite3
import os

# Create database folder
os.makedirs("database", exist_ok=True)

# Create a single database containing all customers
conn = sqlite3.connect("database/single_database.db")
df.to_sql("customers", conn, if_exists="replace", index=False)
conn.close()

# Create three separate shards
for shard_no in range(1, 4):

    # Select records for this shard
    shard_df = df[df["customer_id"] % 3 == shard_no - 1]

    # Create separate database
    conn = sqlite3.connect(
        f"database/shard_{shard_no}.db"
    )

    shard_df.to_sql(
        "customers",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

    print(f"Shard {shard_no} created: {len(shard_df)} records")

print("\nAll three shards created successfully!")

Shard 1 created: 100 records
Shard 2 created: 100 records
Shard 3 created: 100 records

All three shards created successfully!


In [5]:
for shard_no in range(1, 4):

    conn = sqlite3.connect(
        f"database/shard_{shard_no}.db"
    )

    shard_df = pd.read_sql_query(
        "SELECT * FROM customers",
        conn
    )

    conn.close()

    print(f"\nShard {shard_no}")
    print("Total records:", len(shard_df))
    display(shard_df.head(5))


Shard 1
Total records: 100


,customer_id,name,city,email,balance
0,1002,Customer_1002,Nashik,customer1002@gmail.com,10020
1,1005,Customer_1005,Mumbai,customer1005@gmail.com,10050
2,1008,Customer_1008,Pune,customer1008@gmail.com,10080
3,1011,Customer_1011,Nagpur,customer1011@gmail.com,10110
4,1014,Customer_1014,Nashik,customer1014@gmail.com,10140



Shard 2
Total records: 100


,customer_id,name,city,email,balance
0,1003,Customer_1003,Nagpur,customer1003@gmail.com,10030
1,1006,Customer_1006,Nashik,customer1006@gmail.com,10060
2,1009,Customer_1009,Mumbai,customer1009@gmail.com,10090
3,1012,Customer_1012,Pune,customer1012@gmail.com,10120
4,1015,Customer_1015,Nagpur,customer1015@gmail.com,10150



Shard 3
Total records: 100


,customer_id,name,city,email,balance
0,1001,Customer_1001,Mumbai,customer1001@gmail.com,10010
1,1004,Customer_1004,Pune,customer1004@gmail.com,10040
2,1007,Customer_1007,Nagpur,customer1007@gmail.com,10070
3,1010,Customer_1010,Nashik,customer1010@gmail.com,10100
4,1013,Customer_1013,Mumbai,customer1013@gmail.com,10130


In [6]:
def route_query(customer_id):

    shard_no = (int(customer_id) % 3) + 1

    conn = sqlite3.connect(
        f"database/shard_{shard_no}.db"
    )

    result = pd.read_sql_query(
        "SELECT * FROM customers WHERE customer_id = ?",
        conn,
        params=(int(customer_id),)
    )

    conn.close()

    return result, shard_no


# Test query routing
test_ids = [1001, 1002, 1003, 1010, 1025]

for cid in test_ids:

    result, shard = route_query(cid)

    print(
        f"Customer ID: {cid} | "
        f"Selected Shard: {shard} | "
        f"Found: {not result.empty}"
    )

Customer ID: 1001 | Selected Shard: 3 | Found: True
Customer ID: 1002 | Selected Shard: 1 | Found: True
Customer ID: 1003 | Selected Shard: 2 | Found: True
Customer ID: 1010 | Selected Shard: 3 | Found: True
Customer ID: 1025 | Selected Shard: 3 | Found: True


In [7]:
!pip install -q ipywidgets

In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# Interactive controls
customer_input = widgets.IntText(
    value=1001,
    description="Customer ID:"
)

search_button = widgets.Button(
    description="Search Customer",
    button_style="success"
)

shard_dropdown = widgets.Dropdown(
    options=[1, 2, 3],
    value=1,
    description="Select Shard:"
)

show_button = widgets.Button(
    description="Show Shard Data"
)

performance_button = widgets.Button(
    description="Compare Performance",
    button_style="info"
)

output = widgets.Output()


# Search customer
def search_customer(button):

    with output:
        clear_output()

        cid = customer_input.value

        if cid not in df["customer_id"].values:
            print("Customer ID not found!")
            return

        result, shard = route_query(cid)

        print("CUSTOMER SEARCH RESULT")
        print("----------------------")
        print("Customer ID:", cid)
        print("Automatically selected shard:", shard)
        display(result)


# Show selected shard
def show_shard(button):

    with output:
        clear_output()

        shard_no = shard_dropdown.value

        conn = sqlite3.connect(
            f"database/shard_{shard_no}.db"
        )

        shard_data = pd.read_sql_query(
            "SELECT * FROM customers",
            conn
        )

        conn.close()

        print(f"SHARD {shard_no} DETAILS")
        print("Total records:", len(shard_data))
        display(shard_data)


# Compare query performance
def compare_performance(button):

    with output:
        clear_output()

        cid = customer_input.value

        if cid not in df["customer_id"].values:
            print("Customer ID not found!")
            return

        # Single database timing
        conn = sqlite3.connect(
            "database/single_database.db"
        )

        start = time.perf_counter()

        single_result = pd.read_sql_query(
            "SELECT * FROM customers WHERE customer_id = ?",
            conn,
            params=(cid,)
        )

        single_time = (
            time.perf_counter() - start
        ) * 1000

        conn.close()

        # Sharded database timing
        start = time.perf_counter()

        shard_result, shard_no = route_query(cid)

        shard_time = (
            time.perf_counter() - start
        ) * 1000

        performance_df = pd.DataFrame({
            "Database": ["Single Database", "Sharded Database"],
            "Execution Time (ms)": [
                single_time,
                shard_time
            ]
        })

        print("PERFORMANCE COMPARISON")
        print("Customer ID:", cid)
        print("Selected Shard:", shard_no)

        display(performance_df)

        fig = px.bar(
            performance_df,
            x="Database",
            y="Execution Time (ms)",
            title="Single Database vs Sharded Database"
        )

        fig.show()


# Connect buttons
search_button.on_click(search_customer)
show_button.on_click(show_shard)
performance_button.on_click(compare_performance)


# Display interactive sheet
display(
    widgets.VBox([
        widgets.HTML("<h2>Distributed Database Sharding Simulator</h2>"),
        widgets.HTML("<b>1. Customer Query Router</b>"),
        customer_input,
        search_button,
        widgets.HTML("<hr><b>2. Shard Explorer</b>"),
        shard_dropdown,
        show_button,
        widgets.HTML("<hr><b>3. Performance Analysis</b>"),
        performance_button,
        output
    ])
)

In [9]:
!pip install -q streamlit plotly

In [10]:
%%writefile app.py

import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px
import time
import os

st.set_page_config(
    page_title="Database Sharding Simulator",
    page_icon="🗄️",
    layout="wide"
)

st.title("🗄️ Distributed Database Sharding Simulator")
st.caption("Customer Data Distribution and Intelligent Query Routing")

DB_FOLDER = "database"

# Load original dataset
@st.cache_data
def load_data():
    return pd.read_csv("customers_sample_300.csv")

df = load_data()


# Read shard data
def get_shard_data(shard_no):

    conn = sqlite3.connect(
        f"{DB_FOLDER}/shard_{shard_no}.db"
    )

    data = pd.read_sql_query(
        "SELECT * FROM customers",
        conn
    )

    conn.close()

    return data


# Query routing
def route_query(customer_id):

    shard_no = (int(customer_id) % 3) + 1

    conn = sqlite3.connect(
        f"{DB_FOLDER}/shard_{shard_no}.db"
    )

    start = time.perf_counter()

    result = pd.read_sql_query(
        "SELECT * FROM customers WHERE customer_id = ?",
        conn,
        params=(int(customer_id),)
    )

    execution_time = (
        time.perf_counter() - start
    ) * 1000

    conn.close()

    return result, shard_no, execution_time


# Sidebar navigation
st.sidebar.title("Navigation")

page = st.sidebar.radio(
    "Select Section",
    [
        "Dashboard",
        "Shard Explorer",
        "Query Router",
        "Performance Analysis"
    ]
)


# Dashboard
if page == "Dashboard":

    st.header("Project Dashboard")

    counts = [
        len(get_shard_data(i))
        for i in range(1, 4)
    ]

    c1, c2, c3 = st.columns(3)

    c1.metric("Total Customers", len(df))
    c2.metric("Total Shards", 3)
    c3.metric("Records per Shard", 100)

    st.subheader("Customer Distribution")

    distribution = pd.DataFrame({
        "Shard": ["Shard 1", "Shard 2", "Shard 3"],
        "Records": counts
    })

    fig = px.pie(
        distribution,
        names="Shard",
        values="Records",
        title="Records Stored in Each Shard"
    )

    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Customer Dataset")
    st.dataframe(df, use_container_width=True)


# Shard Explorer
elif page == "Shard Explorer":

    st.header("Shard Explorer")

    selected_shard = st.selectbox(
        "Choose a Shard",
        [1, 2, 3]
    )

    shard_data = get_shard_data(selected_shard)

    st.metric("Total Records", len(shard_data))

    st.dataframe(
        shard_data,
        use_container_width=True
    )


# Query Router
elif page == "Query Router":

    st.header("Intelligent Query Routing")

    customer_id = st.number_input(
        "Enter Customer ID",
        min_value=1001,
        max_value=1300,
        step=1
    )

    if st.button("Search Customer"):

        result, shard, execution_time = route_query(
            customer_id
        )

        st.success(
            f"Query automatically routed to Shard {shard}"
        )

        st.metric(
            "Execution Time",
            f"{execution_time:.4f} ms"
        )

        if not result.empty:
            st.subheader("Customer Details")
            st.dataframe(result, use_container_width=True)
        else:
            st.warning("Customer not found!")


# Performance Analysis
elif page == "Performance Analysis":

    st.header("Performance Comparison")

    customer_id = st.number_input(
        "Enter Customer ID for comparison",
        min_value=1001,
        max_value=1300,
        value=1001,
        step=1
    )

    if st.button("Compare Performance"):

        # Single database query
        conn = sqlite3.connect(
            f"{DB_FOLDER}/single_database.db"
        )

        start = time.perf_counter()

        single_result = pd.read_sql_query(
            "SELECT * FROM customers WHERE customer_id = ?",
            conn,
            params=(customer_id,)
        )

        single_time = (
            time.perf_counter() - start
        ) * 1000

        conn.close()

        # Sharded database query
        shard_result, shard, shard_time = route_query(
            customer_id
        )

        performance_df = pd.DataFrame({
            "Database": [
                "Single Database",
                "Sharded Database"
            ],
            "Execution Time (ms)": [
                single_time,
                shard_time
            ]
        })

        st.write("Performance Results")
        st.dataframe(performance_df)

        fig = px.bar(
            performance_df,
            x="Database",
            y="Execution Time (ms)",
            title="Query Execution Time Comparison"
        )

        st.plotly_chart(fig, use_container_width=True)


st.sidebar.markdown("---")
st.sidebar.caption("Distributed Computing Mini Project")

Writing app.py


In [11]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [12]:
!cat streamlit.log



2026-11-20 09:01:04.169 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.32.149.164:8501



In [13]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

cloudflared version 2026.11.1 (built 2026-11-17-15:04 UTC)


In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-11-20T09:01:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-20T09:01:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-20T09:01:44Z INF +--------------------------------------------------------------------------------------------+
2026-11-20T09:01:44Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-20T09:01:44Z INF |  https://physiology-hansen-gig-playstation.trycloudfla